# Store 1 — Customer Data Cleaning & Segmentation

**Author:** Mario Alberto Noriega Barrios  
**Tools:** Python — core data structures, list comprehensions, functions

---

## Business Context

Store 1 is preparing to launch a Customer Loyalty Program and needs its customer database cleaned and structured before any marketing campaign can run. The raw data contains inconsistent formatting — names with extra whitespace, mixed case categories, and floating-point ages — that would compromise any downstream segmentation.

**Objective:** Transform a raw customer dataset into a clean, analysis-ready structure, then answer three key business questions:

1. What is the total revenue generated across all customers?
2. Which young customers (under 30) represent high-value segments?
3. Which customers can be targeted by product category?

---

## Dataset

Each customer record contains five fields:

| Field | Type | Description |
|---|---|---|
| `user_id` | string | Unique customer identifier |
| `user_name` | string | Full name (raw, unformatted) |
| `user_age` | float | Age (needs integer conversion) |
| `fav_categories` | list | Purchase categories (uppercase) |
| `total_spendings` | list | Amount spent per category |

---

## 1. Data Cleaning Pipeline

The raw data presents three issues that must be resolved before analysis:
- Names contain leading/trailing whitespace and underscore separators
- Ages are stored as floats instead of integers
- Categories are uppercase, which causes mismatches in string comparisons

The cleaning logic is encapsulated in a reusable function to ensure consistency across all records.

In [ ]:
users_raw = [
    ['32415', ' mike_reed ', 32.0, ['ELECTRONICS', 'SPORT', 'BOOKS'], [894, 213, 173]],
    ['31980', 'kate morgan', 24.0, ['CLOTHES', 'BOOKS'], [439, 390]],
    ['32156', ' john doe ', 37.0, ['ELECTRONICS', 'HOME', 'FOOD'], [459, 120, 99]],
    ['32761', 'SAMANTHA SMITH', 29.0, ['CLOTHES', 'ELECTRONICS', 'BEAUTY'], [299, 679, 85]],
    ['32984', 'David White', 41.0, ['BOOKS', 'HOME', 'SPORT'], [234, 329, 243]],
    ['33001', 'emily brown', 26.0, ['BEAUTY', 'HOME', 'FOOD'], [213, 659, 79]],
    ['33767', ' Maria Garcia', 33.0, ['CLOTHES', 'FOOD', 'BEAUTY'], [499, 189, 63]],
    ['33912', 'JOSE MARTINEZ', 22.0, ['SPORT', 'ELECTRONICS', 'HOME'], [259, 549, 109]],
    ['34009', 'lisa wilson ', 35.0, ['HOME', 'BOOKS', 'CLOTHES'], [329, 189, 329]],
    ['34278', 'James Lee', 28.0, ['BEAUTY', 'CLOTHES', 'ELECTRONICS'], [189, 299, 579]]
]

In [ ]:
def clean_user(user):
    """
    Normalizes a raw customer record for analysis.

    Transformations applied:
    - Strips whitespace and replaces underscores in name, then splits into [first, last]
    - Converts age from float to integer
    - Lowercases all purchase categories for consistent string matching

    Args:
        user (list): Raw customer record [id, name, age, categories, spendings]

    Returns:
        list: Cleaned record in the same structure
    """
    name = user[1].strip().replace("_", " ").split()
    age = int(user[2])
    categories = [cat.lower() for cat in user[3]]
    return [user[0], name, age, categories, user[4]]


# Apply cleaning to all records using list comprehension
users_clean = [clean_user(user) for user in users_raw]

print(f"Records processed: {len(users_clean)}")
print(f"Sample record: {users_clean[0]}")

---

## 2. Revenue Analysis

Total revenue is calculated by summing all spending lists across the customer base. A generator expression avoids creating an intermediate list in memory, which is more efficient at scale.

In [ ]:
# Generator expression: sum each customer's spendings, then sum all results
revenue = sum(sum(user[4]) for user in users_clean)

print(f"Total revenue across all customers: ${revenue:,}")

---

## 3. Customer Segmentation

### 3.1 Young Customers (Under 30)

The marketing team wants to target customers under 30 for a specific campaign. This segment tends to respond well to digital channels and loyalty incentives.

In [ ]:
print("Customers under 30:")
print("\n".join([user[1][0] for user in users_clean if user[2] < 30]))

### 3.2 High-Value Young Customers

Within the under-30 segment, customers with total spending above $1,000 represent a priority group — they combine youth (long customer lifetime potential) with proven purchasing power.

In [ ]:
print("High-value customers under 30 (spending > $1,000):")
print("\n".join([user[1][0] for user in users_clean if user[2] < 30 and sum(user[4]) > 1000]))

### 3.3 Category-Based Segmentation

The function below enables flexible, reusable segmentation by any product category. This supports targeted campaigns without duplicating logic for each category.

In [ ]:
def get_clients_by_category(users, category):
    """
    Returns all customers who purchased in a given category.

    Args:
        users (list): Cleaned customer records
        category (str): Category to filter by (lowercase)

    Returns:
        list: Filtered records as [id, name, age, total_spending]
    """
    return [
        [user[0], user[1], user[2], sum(user[4])]
        for user in users
        if category in user[3]
    ]


# Example: customers who purchased in the 'home' category
home_buyers = get_clients_by_category(users_clean, 'home')

print(f"Customers in 'home' category: {len(home_buyers)}")
for client in home_buyers:
    print(f"  {client[1][0].capitalize()} {client[1][1].capitalize()} — Age {client[2]} — Total: ${client[3]}")

---

## 4. Key Findings

| Metric | Result |
|---|---|
| Total customers | 10 |
| Total revenue | $9,189 |
| Customers under 30 | 5 |
| High-value customers under 30 (>$1,000) | 2 |
| Customers in 'home' category | 5 |

**Recommendations for the marketing team:**

- **Priority segment:** Samantha and James are under 30 and spending over $1,000 — ideal candidates for a premium loyalty tier with early access or exclusive discounts.
- **Home category campaign:** 5 out of 10 customers purchased in the home category, representing 50% of the base. This category warrants a dedicated campaign.
- **Data quality note:** The raw dataset contained inconsistent name formatting and uppercase categories. A standardized data entry process would reduce future cleaning overhead.